# FSDP / TP / CP：从通信耗时到并行选择

`07.02`–`07.04` 已分别给出 FSDP、经典 TP 与 CP 的原理、通信量和可运行的两卡测量入口。本节不再重复逐项推导，只做三件事：**统一比较口径、说明如何判断通信暴露时间、给出多轮 SFT 的验证条件。**

所有 MB/GB 均为十进制单位。独立 collective 的慢 rank latency 从各 notebook 生成的 JSON 读取；其串行和只叫“零 overlap 串行估算”，不是训练 step time 或系统上界。最终结论必须由相同 workload 的 tokens/s、峰值显存和跨 rank trace 验证。


## 1. 比较前先统一口径

设通信组大小为 $p$，$M$ 是 all-gather、reduce-scatter、all-reduce 对应的完整逻辑张量，$M_{local}$ 是 all-to-all 前每个 rank 的本地输入。本节只报告**每 rank 单向发送 payload**；均匀通信时接收量大致相同，发送加接收约为表中数字的两倍。

| collective | 每 rank 逻辑输入 → 输出 | ring/pairwise 基线下每 rank 发送量 |
|---|---|---:|
| all-gather | $M/p \rightarrow M$ | $\frac{p-1}{p}M$ |
| reduce-scatter | $M \rightarrow M/p$ | $\frac{p-1}{p}M$ |
| all-reduce | $M \rightarrow M$ | $2\frac{p-1}{p}M$ |
| all-to-all | $M_{local} \rightarrow M_{local}$ | $\frac{p-1}{p}M_{local}$ |

逻辑张量、链路 payload、发送加接收和 profiler 的 `Transit Size` 不是同一个量，不能混用。通信时间还包含启动、同步、拓扑和资源竞争，可用 $T_{comm}\approx k\alpha+V_{send}/BW_{eff}$ 判断趋势，但不能仅凭 GB 排出速度名次。


## 2. Qwen3-1.7B 两卡结果摘要

固定 28 个 Transformer blocks、bf16、`B=2, S=4096`，三行分别是 degree 2 的**独立基线**：

| 模式 | 一层 forward + backward 的教学账本 | 28 层每 rank 单向发送 | latency 来源 | 详细分析 |
|---|---:|---:|---|---|
| FSDP（每层一 unit、reshard=True） | 2 AG + 1 RS | 4.228 GB | `results/fsdp_latest.json` | [07.02 §3–§4](07.02_fsdp_collectives.ipynb) |
| 经典 TP（SP 关闭） | 4 AR | 3.758 GB | `results/tp_classic_latest.json` | [07.03 §3、§5](07.03_tp_collectives.ipynb) |
| Ulysses-CP（未融合 Q/K/V） | forward/backward 共 8 A2A | 1.409 GB | `results/cp_latest.json` | [07.04 §4、§6](07.04_cp_collectives.ipynb) |

这张表只用于核对公式，不能直接宣布哪条路线最快：它混合了 FSDP、经典 TP 和 Ulysses-CP 三种不同数据归属。当前 TorchTitan 默认 TP+SP 的实际 AG/RS 账本甚至不在这张经典 TP 行里。FSDP rank 通常处理不同样本，TP/CP 组内 rank 共同处理同一批 token；调用依赖、overlap 和额外布局变换也不同。


## 3. Overlap：隐藏时间，不减少通信量

局部通信区间可先用下面两个量描述：

$$
r_{overlap}=\frac{T_{comm\cap compute}}{T_{comm}},\qquad
T_{exposed}\approx T_{comm}-T_{comm\cap compute}.
$$

| 模式 | 主要 overlap 窗口 | 常见暴露部分 | 代价 |
|---|---|---|---|
| FSDP | 当前层计算与下一层参数 AG；反向计算与 RS | 首个 AG、末尾 RS、未及时完成的预取 | 完整参数与通信 buffer 的峰值显存 |
| 经典 TP / TP+SP | 经典 AR 的独立工作；TP+SP 的分块 AG→GEMM 或 GEMM→RS、独立 wgrad | 下游要求目标布局时的 wait | GEMM 被切碎，通信与计算争用资源 |
| CP | Q/K/V 分支、跨层或分块 attention | pre-attention A2A 与输出布局恢复 | reshape/transpose buffer 与 head 整除约束 |

高 overlap 比例不一定带来相同比例的端到端收益：首尾 bubble、rank skew、多个 communicator 竞争都可能改变关键路径。最高可信度指标是固定 workload 后对比 overlap 开/关的 median step time 或 tokens/s。


## 4. Qwen3-1.7B 多轮 SFT：为什么候选是 FSDP 或 CP

“多轮”本身不是并行维度。history token 即使不参与 loss，也仍参与前向、反向和激活通信；真正影响选型的是本地 token 数、单条序列长度、模型状态显存和 rank 间负载均衡。

- **模型状态或独立样本吞吐是瓶颈：选 FSDP。** 它分片参数、梯度和优化器状态；参数通信基本固定，单 rank 能容纳的 token 越多，通信/token 越容易被摊薄，且逐层 AG/RS 有较好的预取窗口。但 FSDP 不切分单条样本的 attention 激活。
- **单条长序列的激活或 attention 是瓶颈：选 CP=2。** 两个 rank 共同处理同一序列；Qwen3 的 16 个 Q heads 和 8 个 KV heads都能被 2 整除。它能解决 FSDP 无法解决的单样本切分问题，但要支付 pre-attention A2A 的硬等待和小消息开销。
- **模型宽度、单层 GEMM 或权重分片是瓶颈：评估 TP+SP。** 当前 TorchTitan 默认 SP 会让部分 activation 沿 sequence 分片，并用 AG/RS 完成模块边界布局转换；不能用经典每层 4 次 AR 的基线替代实际账本。

字节账本可以计算通信量/token 的交点，但它不是性能胜负线：overlap、A2A 等待、AG/RS 布局转换、拓扑、显存和 workload 所有权都可能反转排序。实际决策应先指出瓶颈，再对候选路线运行相同 workload。

本节先形成两个理论结论：第一，FSDP 的逐层 all-gather/reduce-scatter 存在与相邻层计算重叠的窗口，因此在通信暴露时间上具有结构性优势；第二，CP 能把单条长序列分摊到多个 rank，虽然引入 attention 前后的 all-to-all，但在长序列内存受限场景下，其通信量是可核算、可接受的容量代价。这里不把它们写成已验证的端到端性能排名，下一章再用统一的 2NPU workload 和 profiler 验证。


## 5. 下一章导读：从理论结论到真实训练证据

本节只建立通信账本和并行选择的理论边界，不在这里运行 SFT 或给出端到端性能排名。下一章将把本节的两个结论放回统一的 2NPU Wordle SFT workload 中验证：FSDP 的 all-gather/reduce-scatter 是否能与相邻计算重叠，以及 CP 的 all-to-all 是否以可接受的通信代价支持更长序列。

验证会固定 global batch、micro-batch、sequence、packing、activation checkpoint、dtype、compile 和 mesh，并同时保留所有 rank 的 trace。分析时分别查看 Stage、Computing、通信暴露、等待/同步、reshape/copy、tokens/s 和峰值显存，避免用理论 payload 或单个 collective latency 代替训练结论。

阅读路径：在 [08.03](../08_comm_optimizations/08.03_cp_long_sequence_capacity_profiling.ipynb) 完成 CP 的长序列容量实验与 trace 分析，再用 [08.04](../08_comm_optimizations/08.04_tp2_fsdp2_comparison.ipynb) 做 TP2/FSDP2 同 workload 对比，最后完成 [08.05](../08_comm_optimizations/08.05_chapter_practice.ipynb) 练习。

## 小结

- 各模式的 Qwen3 字节账本与可运行测量入口分别放在 07.02、07.03、07.04；latency 由逐 rank JSON 自动生成。
- overlap 只减少暴露时间，不减少 payload；异步 API 也不会自动消除数据依赖。
- FSDP 解决模型状态与独立样本并行，CP 分摊单条长上下文 attention，TP+SP 分片模型宽度并维持部分 sequence-sharded activation；三者边界不能用一张经典 AR 表代替。
- 任何理论选择都必须用相同 workload 的端到端吞吐、峰值显存和跨 rank trace 验证。


## References

- [Qwen3-1.7B config.json](https://huggingface.co/Qwen/Qwen3-1.7B/blob/b9352fbb8ce704292730cf54b3b1dceb2a808738/config.json)
- [PyTorch `fully_shard`](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html)
- [PyTorch Tensor Parallel tutorial](https://docs.pytorch.org/tutorials/intermediate/TP_tutorial.html)
- [DeepSpeed Ulysses](https://arxiv.org/abs/2309.14509)
- [PyTorch Holistic Trace Analysis](https://docs.pytorch.org/tutorials/beginner/hta_intro_tutorial.html)


## 练习

1. （判断题）通信 payload、collective latency 和最终暴露在 step 关键路径上的时间是三个不同口径。

2. （判断题）Overlap 可以减少暴露时间，但不会减少 collective 实际交换的字节数。

3. （单选题）为什么不能把各个独立 collective 的 median 串行相加称为系统最坏情况上界？
    A. 真实训练的调用顺序、rank skew、等待和 overlap 会改变关键路径
    B. collective 没有任何耗时
    C. profiler 无法记录通信
    D. payload 与 world size 无关

4. （多选题）选择 FSDP、TP 或 CP 前应核对哪些证据？
    A. 内存约束与实际 OOM
    B. resolved parallel config
    C. 双 rank trace 与 normalized step 指标
    D. 只比较理论通信量最小值

In [ ]:
!cat ./answer/07.05_answer.txt
